# RAG 第 18 课：Conversational RAG（多轮对话 RAG）

前 17 课里，我们都默认每次查询是**独立的一句完整问题**。
但真实产品里用户很少这么乖：

- 第一轮：`Who founded SpaceX?`
- 第二轮：`when was it founded?`  ← "it" 指代
- 第三轮：`and who is the current CTO?`  ← 没有主语

如果直接拿第二、三轮去检索，会灾难性失败。

这一课我们加一步：**Contextual Query Reformulation**

也就是利用历史对话，把当前这一句改写成**一个独立可检索的完整问题**，再走正常的 RAG 流程。

In [ ]:
import gc
try:
    import torch
    if torch.cuda.is_available():
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print('GPU cleared.')
    else: print('CUDA not available.')
except Exception as e: print(f'no torch ({e})')

In [ ]:
import math, re, json
from collections import Counter, defaultdict
from typing import List, Tuple
import numpy as np
from datasets import load_dataset
from openai import OpenAI

client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
preferred = ['qwen/qwen3.6-35b-a3b', 'qwen3.6-35b-a3b', 'qwen3.5-35b-a3b', 'qwen3.5-27b']
chat_model = next((m for m in preferred if m in model_ids), None)
embedding_model = next((m for m in model_ids if 'embed' in m.lower()), None)
print('chat =', chat_model, '| embed =', embedding_model)

## 1. 数据 + 检索层（精简版）

In [ ]:
raw_ds = load_dataset('squad', split='validation[:160]')
documents=[]; seen={}
for item in raw_ds:
    c = item['context'].strip()
    if c not in seen:
        seen[c] = len(documents)
        documents.append({'doc_id': len(documents), 'text': c, 'title': item['title']})

def tokenize(t): return re.findall(r'[a-zA-Z0-9]+', t.lower())

class BM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.dl = np.array([len(t) for t in corpus], dtype=np.float32)
        self.avgdl = float(self.dl.mean()); self.N = len(corpus)
        self.post = defaultdict(dict)
        for did,toks in enumerate(corpus):
            for tok,f in Counter(toks).items(): self.post[tok][did]=f
        self.idf = {t: math.log(1+(self.N-len(p)+0.5)/(len(p)+0.5)) for t,p in self.post.items()}
    def score(self, qt):
        s=np.zeros(self.N, dtype=np.float32)
        for tok in qt:
            if tok not in self.post: continue
            idf=self.idf[tok]
            for did,f in self.post[tok].items():
                dl=self.dl[did]
                s[did]+=idf*(f*(self.k1+1))/(f+self.k1*(1-self.b+self.b*dl/self.avgdl))
        return s

bm25 = BM25([tokenize(d['text']) for d in documents])

def get_emb(texts, bs=16):
    out=[]
    for i in range(0,len(texts),bs):
        r=client.embeddings.create(model=embedding_model, input=texts[i:i+bs])
        out.extend([np.array(x.embedding, dtype=np.float32) for x in r.data])
    return np.vstack(out)
def l2n(m):
    n=np.linalg.norm(m,axis=1,keepdims=True); return m/np.clip(n,1e-12,None)
doc_embeddings = l2n(get_emb([d['text'] for d in documents]))

def hybrid(q, top_k=3, cand=15):
    lex=np.argsort(bm25.score(tokenize(q)))[::-1][:cand]
    qe=l2n(get_emb([q]))[0]; den=np.argsort(doc_embeddings@qe)[::-1][:cand]
    fused=defaultdict(float)
    for rk,i in enumerate(lex,1): fused[int(i)]+=1/(60+rk)
    for rk,i in enumerate(den,1): fused[int(i)]+=1/(60+rk)
    return [d for d,_ in sorted(fused.items(),key=lambda x:x[1],reverse=True)[:top_k]]
print('docs =', len(documents))

## 2. 关键一步：History-Aware Query Reformulation

给 LLM 看"最近几轮对话 + 当前这一句"，让它输出一个**独立可检索**的问题。

prompt 核心约束：
- 不要回答问题
- 不要加事实
- 补全指代、补全主语
- 如果当前这一句已经自足，就原样返回

In [ ]:
def reformulate_query(history: List[Tuple[str,str]], current: str) -> str:
    if not history:
        return current
    hist_block = '\n'.join([f'User: {u}\nAssistant: {a}' for u,a in history[-3:]])
    sys = ('You rewrite the user\'s latest message into a self-contained search query '
           'using the conversation history as context. '
           'Rules: (1) resolve all pronouns and references; '
           '(2) add any missing subjects; '
           '(3) do not answer the question; '
           '(4) do not invent facts that are not in the history or current message; '
           '(5) if the latest message is already self-contained, return it unchanged; '
           '(6) output the rewritten query only.')
    usr = f'Conversation so far:\n{hist_block}\n\nLatest user message: {current}\n\nSelf-contained query:'
    r = client.chat.completions.create(model=chat_model, temperature=0,
        messages=[{'role':'system','content':sys},{'role':'user','content':usr}])
    return r.choices[0].message.content.strip().strip('"').strip("'")

## 3. 多轮对话的回答函数

流程：
1. 拿历史 + 当前消息 → reformulate
2. reformulated query → hybrid 检索
3. 把 context + 完整对话历史都喂给 LLM 生成答案

注意第 3 步：生成时还是要把**历史对话**放给 LLM，因为用户的问法本身在历史里，这样回答更自然。

In [ ]:
def build_context(doc_ids):
    return '\n\n'.join([f'[Document {i}] title: {documents[d]["title"]}\n{documents[d]["text"]}' for i,d in enumerate(doc_ids,1)])

def conversational_answer(history, current, use_reformulation=True, top_k=3):
    search_q = reformulate_query(history, current) if use_reformulation else current
    doc_ids = hybrid(search_q, top_k=top_k)
    ctx = build_context(doc_ids)
    sys = ('You are a careful assistant. Answer the user\'s latest message using the provided context and the conversation history. '
           'Only use facts from the context. If the context does not support the answer, say you do not know. Keep it short.')
    msgs = [{'role':'system','content':sys}]
    for u,a in history:
        msgs.append({'role':'user','content':u})
        msgs.append({'role':'assistant','content':a})
    msgs.append({'role':'user','content':f'{current}\n\nContext:\n{ctx}'})
    r = client.chat.completions.create(model=chat_model, temperature=0, messages=msgs)
    return r.choices[0].message.content.strip(), search_q, doc_ids

## 4. 对比：有 reformulation vs 没有

我们手动造一段多轮对话，核心是**第二轮带指代**，看两种策略哪一个检索得对。

In [ ]:
# 挑一条 squad 里能用的问题做第一轮
raw_item = next(x for x in raw_ds if x['answers']['text'])
first_q = raw_item['question']
first_title = raw_item['title']
print('topic =', first_title)
print('turn 1 query =', first_q)

# 假设第二轮是个带指代的短 query
turn2 = 'and who led it?'
turn3 = 'when did that happen?'

history = []
# turn 1 走一遍
a1, sq1, ids1 = conversational_answer(history, first_q)
print('\n--- turn 1 ---')
print('search_q :', sq1)
print('doc_ids  :', ids1)
print('answer   :', a1)
history.append((first_q, a1))

In [ ]:
print('=== turn 2: WITHOUT reformulation ===')
a2_no, sq2_no, ids2_no = conversational_answer(history, turn2, use_reformulation=False)
print('search_q :', sq2_no)
print('doc_ids  :', ids2_no)
print('answer   :', a2_no)

print('\n=== turn 2: WITH reformulation ===')
a2, sq2, ids2 = conversational_answer(history, turn2, use_reformulation=True)
print('search_q :', sq2)
print('doc_ids  :', ids2)
print('answer   :', a2)
history.append((turn2, a2))

In [ ]:
print('=== turn 3: WITHOUT reformulation ===')
a3_no, sq3_no, ids3_no = conversational_answer(history, turn3, use_reformulation=False)
print('search_q :', sq3_no)
print('doc_ids  :', ids3_no)
print('answer   :', a3_no)

print('\n=== turn 3: WITH reformulation ===')
a3, sq3, ids3 = conversational_answer(history, turn3, use_reformulation=True)
print('search_q :', sq3)
print('doc_ids  :', ids3)
print('answer   :', a3)

## 5. 工程直觉

1. **没有 reformulation 的多轮 RAG 基本是坏的**。只有第一轮能工作。
2. reformulation 的 prompt 一定要加"不要回答" + "不要编造事实"两条硬约束，否则 LLM 会自己发挥。
3. 历史只看最近 3 轮通常够用。更长的历史反而会引入歧义。
4. 历史里如果出现**话题切换**（用户突然问不相关的事），reformulation 容易错。可以用一个先验分类器先判断"是否延续上一话题"。

对话状态（history）属于"会话级记忆"，真实产品里通常存在 Redis 或数据库里，按 session_id 隔离。

下一课：**Citations & Grounding**，让 LLM 生成答案时**逐句标注来源 doc_id**，用户可以点击回原文校验。